# 使用 Kaggle Internet + GPU 生成 MiniOneRec 商品文本 Embedding

在 Kaggle Notebook 的 Settings 中同时启用 **GPU** 与 **Internet**。本 notebook 直接使用 Hugging Face 模型 ID 下载 Qwen，不需要预先上传模型文件。最后一个运行单元会生成 `Industrial_and_Scientific.emb-qwen-td.npy`。

In [1]:
# 安装运行 Qwen2.5 与 Accelerate 所需依赖。
# 安装完成后若 Kaggle 提示重启 kernel，请重启后从第一个单元重新运行。
!pip install -q -U 'transformers>=4.43.1' 'accelerate>=0.30.0' 'huggingface_hub>=0.24.0' safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 82.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.1 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# 此单元只检查 Kaggle 是否正确分配了 GPU。
import torch

if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU。请在 Kaggle Notebook Settings 中将 Accelerator 设为 GPU。')

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f'GPU: {gpu_name}')
print(f'显存: {gpu_memory_gib:.2f} GiB')
print(f'PyTorch CUDA: {torch.version.cuda}')

GPU: Tesla T4
显存: 14.56 GiB
PyTorch CUDA: 12.8


In [3]:
# 从 GitHub 获取运行 text2emb 所需的仓库代码。
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/kaggle/working/MiniOneRec-main")
REPO_URL = "https://github.com/AkaliKong/MiniOneRec.git"

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )

print(f"项目代码目录: {PROJECT_ROOT}")

Cloning into '/kaggle/working/MiniOneRec-main'...


项目代码目录: /kaggle/working/MiniOneRec-main


In [4]:
# 配置 Kaggle Input、Working、项目代码和 Hugging Face 模型。
from pathlib import Path
import os
import shutil

PROJECT_ROOT = Path("/kaggle/working/MiniOneRec-main")
DATASET = "Industrial_and_Scientific"

# 自动在所有 Kaggle Input 中寻找预处理生成的 item.json。
# 这样不依赖 Kaggle Dataset 的具体挂载目录名称。
input_item_files = list(
    Path("/kaggle/input").rglob(f"{DATASET}.item.json")
)

if len(input_item_files) != 1:
    raise FileNotFoundError(
        "未找到唯一的 item.json，请检查 Kaggle Input 是否已正确挂载。\n"
        f"当前找到: {input_item_files}"
    )

INPUT_ITEM_FILE = input_item_files[0]

# Input 是只读的，因此将 item.json 复制到 Working 后再生成 embedding。
WORK_DATA_DIR = (
    Path("/kaggle/working")
    / "minionerec-text2emb-data"
    / DATASET
)
WORK_DATA_DIR.mkdir(parents=True, exist_ok=True)

ITEM_FILE = WORK_DATA_DIR / f"{DATASET}.item.json"
shutil.copy2(INPUT_ITEM_FILE, ITEM_FILE)

# Internet-on 时直接传入 Hugging Face 模型 ID。
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

TEXT2EMB_DIR = PROJECT_ROOT / "rq" / "text2emb"
SOURCE_SCRIPT = TEXT2EMB_DIR / "amazon_text2emb.py"
KAGGLE_SCRIPT = TEXT2EMB_DIR / "amazon_text2emb_kaggle.py"

# embedding 将保存到 Kaggle Working，而不是只读 Input。
DATA_DIR = WORK_DATA_DIR
OUTPUT_FILE = WORK_DATA_DIR / f"{DATASET}.emb-qwen-td.npy"

# Kaggle 单卡与 Qwen 1.5B 的保守起始配置。
NUM_PROCESSES = 1
BATCH_SIZE = 32
MAX_SENT_LEN = 512
ALLOW_OVERWRITE = False

# 模型下载缓存和 CUDA 设置。
os.environ["HF_HOME"] = "/kaggle/working/huggingface-cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Input item.json: {INPUT_ITEM_FILE}")
print(f"Working item.json: {ITEM_FILE}")
print(f"Embedding 输出: {OUTPUT_FILE}")

Input item.json: /kaggle/input/datasets/alettatta/minionerec-datapreparation-text2emb/Industrial_and_Scientific.item.json
Working item.json: /kaggle/working/minionerec-text2emb-data/Industrial_and_Scientific/Industrial_and_Scientific.item.json
Embedding 输出: /kaggle/working/minionerec-text2emb-data/Industrial_and_Scientific/Industrial_and_Scientific.emb-qwen-td.npy


In [5]:
# 此单元检查项目、数据、Hugging Face 网络访问与输出路径；尚未下载完整模型。
from huggingface_hub import HfApi
import json

required_paths = [PROJECT_ROOT, DATA_DIR, TEXT2EMB_DIR, SOURCE_SCRIPT, ITEM_FILE]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        '请检查以下 Kaggle 路径：\n' + '\n'.join(str(path) for path in missing_paths)
    )

if OUTPUT_FILE.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(
        f'embedding 已存在：{OUTPUT_FILE}\n'
        '若确认要重跑，请将 ALLOW_OVERWRITE 改为 True。'
    )

try:
    model_info = HfApi().model_info(MODEL_ID)
except Exception as error:
    raise ConnectionError(
        '无法访问 Hugging Face 模型。请确认 Kaggle Settings 中 Internet 已开启。'
    ) from error

with ITEM_FILE.open('r', encoding='utf-8') as file:
    item_features = json.load(file)

print('输入检查通过。')
print(f'商品数: {len(item_features):,}')
print(f'item.json: {ITEM_FILE}')
print(f'Hugging Face 模型: {model_info.id}')
print(f'预期输出: {OUTPUT_FILE}')
print(f'最大文本长度: {MAX_SENT_LEN}')
print(f'GPU 进程数: {NUM_PROCESSES}')
print(f'Embedding batch size: {BATCH_SIZE}')

输入检查通过。
商品数: 3,106
item.json: /kaggle/working/minionerec-text2emb-data/Industrial_and_Scientific/Industrial_and_Scientific.item.json
Hugging Face 模型: Qwen/Qwen2.5-1.5B-Instruct
预期输出: /kaggle/working/minionerec-text2emb-data/Industrial_and_Scientific/Industrial_and_Scientific.emb-qwen-td.npy
最大文本长度: 512
GPU 进程数: 1
Embedding batch size: 32


In [6]:
# 在 Kaggle 工作副本中生成一个小 batch 的脚本副本，不修改原始 amazon_text2emb.py。
source_code = SOURCE_SCRIPT.read_text(encoding='utf-8')
source_line = 'batch_size = 1024'
target_line = f'batch_size = {BATCH_SIZE}'

if source_line not in source_code:
    raise RuntimeError(
        f'未在官方脚本中找到 {source_line!r}，请检查仓库版本后再运行。'
    )

patched_code = source_code.replace(source_line, target_line, 1)
KAGGLE_SCRIPT.write_text(patched_code, encoding='utf-8')

print(f'已生成 Kaggle 专用脚本：{KAGGLE_SCRIPT}')
print(f'已将内部 batch size 从 1024 改为 {BATCH_SIZE}。')

已生成 Kaggle 专用脚本：/kaggle/working/MiniOneRec-main/rq/text2emb/amazon_text2emb_kaggle.py
已将内部 batch size 从 1024 改为 32。


In [7]:
import torch

print("CUDA 是否可用：", torch.cuda.is_available())
print("GPU 数量：", torch.cuda.device_count())

if torch.cuda.is_available():
    print("当前 GPU：", torch.cuda.current_device())
    print("GPU 名称：", torch.cuda.get_device_name(0))
    print("CUDA 版本：", torch.version.cuda)

if torch.cuda.is_available():
    print("已分配显存：", torch.cuda.memory_allocated(0) / 1024**3, "GB")
    print("已缓存显存：", torch.cuda.memory_reserved(0) / 1024**3, "GB")
    total = torch.cuda.get_device_properties(0).total_memory
    print("总显存：", total / 1024**3, "GB")

CUDA 是否可用： True
GPU 数量： 2
当前 GPU： 0
GPU 名称： Tesla T4
CUDA 版本： 12.8
已分配显存： 0.0 GB
已缓存显存： 0.0 GB
总显存： 14.56219482421875 GB


In [ ]:
import sys

# 使用单张 Kaggle GPU 直接运行脚本。
# 脚本内部的 Accelerator 会自动选择 cuda 设备。

command = [
    sys.executable,
    str(KAGGLE_SCRIPT),
    "--dataset",
    DATASET,
    "--root",
    str(DATA_DIR),
    "--plm_name",
    "qwen",
    "--plm_checkpoint",
    MODEL_ID,
    "--max_sent_len",
    str(MAX_SENT_LEN),
]

print("将执行命令：")
print(" ".join(command))

try:
    subprocess.run(
        command,
        cwd=TEXT2EMB_DIR,
        check=True,
    )
except subprocess.CalledProcessError as error:
    print(f"\nEmbedding 脚本失败，退出码: {error.returncode}")
    print("请查看本单元中该提示之前的完整 Python traceback。")
    raise

if not OUTPUT_FILE.is_file():
    raise RuntimeError(f"脚本已结束，但未找到预期输出：{OUTPUT_FILE}")

print(f"Embedding 已生成：{OUTPUT_FILE}")
print(f"文件大小：{OUTPUT_FILE.stat().st_size / 1024 / 1024:.2f} MiB")

将执行命令：
/usr/bin/python3 /kaggle/working/MiniOneRec-main/rq/text2emb/amazon_text2emb_kaggle.py --dataset Industrial_and_Scientific --root /kaggle/working/minionerec-text2emb-data/Industrial_and_Scientific --plm_name qwen --plm_checkpoint Qwen/Qwen2.5-1.5B-Instruct --max_sent_len 512


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Running with 1 processes.
Process text data: 
Dataset:  Industrial_and_Scientific
args.root:  /kaggle/working/minionerec-text2emb-data/Industrial_and_Scientific
Loading Qwen Model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 207.66it/s]


Total items: 3106
Start generating embeddings with 1 processes...


Proc 0:  88%|████████▊ | 2720/3106 [03:33<00:30, 12.81it/s]

In [9]:
# 生成完成后运行此单元，检查 embedding 的形状、类型、有限值与少量数值。
import numpy as np

embeddings = np.load(OUTPUT_FILE, mmap_mode='r')

print('=== Embedding 校验 ===')
print(f'shape: {embeddings.shape}')
print(f'dtype: {embeddings.dtype}')
print(f'商品数是否对齐: {embeddings.shape[0] == len(item_features)}')
print(f'是否全部为有限值: {np.isfinite(embeddings).all()}')
print('第一个商品 embedding 的前 10 维:')
print(embeddings[0, :10])

=== Embedding 校验 ===
shape: (3106, 1536)
dtype: float32
商品数是否对齐: True
是否全部为有限值: True
第一个商品 embedding 的前 10 维:
[-0.9993835  -0.01257921  0.46023062 -0.40369266 -1.3136716  -1.4598945
 -0.43738458 -1.4273566  -1.6475962  -0.34175244]


生成完成后，请在 Kaggle 中使用 Save Version 或保存输出文件；`/kaggle/working` 中的内容在会话结束后不会永久保留。